# VoxTriad · Notebook 01 – Baseline ML Pipeline
## End-to-End Parkinsonian Speech Classification

**Architecture overview:**

```
Raw .wav  ──▶  [Stage 1] PPD-Net Denoising
                            │
               ┌────────────┴────────────┐
               ▼                         ▼
     [Stage 2a] Classical          [Stage 2b] Deep
     Feature Extraction            Embedding (wav2vec2)
     MFCCs · F0 · Centroid         Mean-pooled hidden states
     Flux · Shimmer proxies        (768-dim)
               └────────────┬────────────┘
                            ▼
               [Stage 3] Tri-Algorithmic
               Meta-Ensemble (soft voting)
               SVC · RF · GBM
                            │
               ┌────────────┴────────────┐
               ▼                         ▼
         Prediction               Epistemic Uncertainty
         (PD / Healthy)           (entropy-based)
```

| Stage | Component | Phase status |
|---|---|---|
| 1 | Spectral-gate denoising | ✅ Implemented |
| 2a | Classical feature extraction | ✅ Implemented |
| 2b | wav2vec2 deep embeddings | ✅ Implemented |
| 3 | Soft-voting meta-ensemble | ✅ Implemented |
| 4 | LOSO-CV validation | ✅ Implemented |
| 5 | Supabase pgvector storage | 🔲 Phase 3 |
| 6 | Gemini LLM report synthesis | 🔲 Phase 3 |

---
## 0 · Environment Setup & Imports

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import os
import pathlib
import warnings
import time
from typing import Dict, List, Optional, Tuple

warnings.filterwarnings('ignore')  # suppress librosa / torch deprecation noise

# ── Numerical ─────────────────────────────────────────────────────────────────
import numpy as np
import scipy.signal
import scipy.stats

# ── Audio I/O & feature extraction ───────────────────────────────────────────
import librosa
import librosa.display
import soundfile as sf

# ── Machine learning ──────────────────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    VotingClassifier,
)
from sklearn.model_selection import LeaveOneGroupOut, cross_val_predict
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

# ── Deep learning ─────────────────────────────────────────────────────────────
import torch
from transformers import Wav2Vec2Processor, Wav2Vec2Model

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({
    'figure.dpi': 130,
    'figure.facecolor': '#0f172a',
    'axes.facecolor': '#1e293b',
    'axes.edgecolor': '#334155',
    'axes.labelcolor': '#cbd5e1',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'text.color': '#f1f5f9',
    'grid.color': '#334155',
    'grid.alpha': 0.4,
    'font.family': 'sans-serif',
})

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Device (MPS → CUDA → CPU) ─────────────────────────────────────────────────
if torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda')
else:
    DEVICE = torch.device('cpu')

print('=' * 60)
print('VoxTriad · Phase 2 ML Pipeline')
print('=' * 60)
print(f'  torch       : {torch.__version__}')
print(f'  librosa     : {librosa.__version__}')
print(f'  compute     : {DEVICE}')
print('=' * 60)

ModuleNotFoundError: No module named 'numpy'

: 

---
## 1 · Stage 1 — PPD-Net Denoising

Simulates the **Pitch-Preserving Denoising Network** (PPD-Net) pre-processing stage.
Implementation uses **spectral gating** — a classic but effective baseline that:

1. Estimates a noise floor from a silent reference segment (first 0.5 s)
2. Computes the STFT of the full signal
3. Applies a soft-threshold mask to suppress frequency bins below the noise profile
4. Reconstructs via iSTFT

Phase 3 will replace this with a learned denoising network.

In [ ]:
def apply_denoising(
    audio_path: str,
    sr_target: int = 16_000,
    n_fft: int = 1024,
    hop_length: int = 256,
    noise_ref_duration_s: float = 0.5,
    prop_decrease: float = 1.0,
    smoothing_factor: int = 3,
) -> Tuple[np.ndarray, int]:
    """
    Load a WAV file and apply spectral-gate noise reduction.

    Simulates the PPD-Net pre-processing stage of VoxTriad.
    Audio is resampled to `sr_target` Hz and converted to mono.

    Args:
        audio_path:           Path to the input .wav file.
        sr_target:            Target sample rate in Hz (default 16 kHz).
        n_fft:                FFT window size.
        hop_length:           STFT hop size.
        noise_ref_duration_s: Duration (s) of leading silence used to
                              estimate the noise floor.
        prop_decrease:        Proportion of noise reduction [0, 1].
        smoothing_factor:     Frequency-bin smoothing kernel width.

    Returns:
        Tuple of (denoised_waveform, sample_rate).

    Raises:
        FileNotFoundError: If `audio_path` does not exist.
    """
    if not pathlib.Path(audio_path).exists():
        raise FileNotFoundError(f'Audio file not found: {audio_path}')

    # 1. Load & resample to mono 16 kHz
    y, sr = librosa.load(audio_path, sr=sr_target, mono=True)

    # 2. Estimate noise floor from the reference segment
    n_noise_samples = int(noise_ref_duration_s * sr)
    noise_clip = y[:n_noise_samples] if len(y) > n_noise_samples else y

    # 3. Compute STFTs
    D_full  = librosa.stft(y,          n_fft=n_fft, hop_length=hop_length)
    D_noise = librosa.stft(noise_clip,  n_fft=n_fft, hop_length=hop_length)

    # Magnitude spectra
    mag_full  = np.abs(D_full)
    mag_noise = np.abs(D_noise)

    # 4. Noise profile: mean + 1σ over noise clip time frames
    noise_mean = mag_noise.mean(axis=1, keepdims=True)  # (freq, 1)
    noise_std  = mag_noise.std(axis=1,  keepdims=True)
    noise_thresh = noise_mean + noise_std

    # 5. Optional frequency-bin smoothing of the threshold
    kernel = np.ones((smoothing_factor, 1)) / smoothing_factor
    noise_thresh_smooth = scipy.signal.fftconvolve(
        noise_thresh, kernel, mode='same'
    )

    # 6. Soft-gate mask:  sigmoid-based smooth threshold
    margin = mag_full - noise_thresh_smooth
    # Sigmoid slope controls the transition sharpness
    mask = 1.0 / (1.0 + np.exp(-10.0 * (margin / (noise_thresh_smooth + 1e-8))))
    mask = mask * prop_decrease + (1.0 - prop_decrease)

    # 7. Apply mask & reconstruct via iSTFT
    D_denoised = D_full * mask
    y_denoised = librosa.istft(D_denoised, hop_length=hop_length, length=len(y))

    return y_denoised, sr


def demo_denoising_on_synthetic() -> Tuple[np.ndarray, int]:
    """
    Demonstrate denoising on a synthetic sustained vowel + pink noise signal.

    Returns:
        Denoised waveform and its sample rate.
    """
    SR = 16_000
    t  = np.linspace(0, 3.0, 3 * SR)          # 3-second signal

    # Simulate a sustained /a/ vowel (fundamental + 4 harmonics)
    f0_hz = 120.0
    vowel = sum(
        (1.0 / k) * np.sin(2 * np.pi * k * f0_hz * t)
        for k in range(1, 6)
    )
    vowel /= np.abs(vowel).max()

    # Add pink-ish noise (1/f spectrum)
    white_noise = np.random.randn(len(t))
    # Simple 1/f approximation via cumulative sum
    pink_noise  = np.cumsum(white_noise)
    pink_noise  = pink_noise / np.abs(pink_noise).max() * 0.15   # SNR ~18 dB

    noisy = vowel + pink_noise

    # Write to a temp file so apply_denoising can load it
    tmp_path = '/tmp/voxtriad_demo.wav'
    sf.write(tmp_path, noisy.astype(np.float32), SR)

    y_clean, sr = apply_denoising(tmp_path)

    # ── Visualise ─────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(2, 2, figsize=(14, 6))
    fig.suptitle('Stage 1 · PPD-Net Denoising (Spectral Gating)', fontsize=13, fontweight='bold', y=1.01)

    t_axis = np.linspace(0, len(noisy) / SR, len(noisy))

    axes[0, 0].plot(t_axis, noisy,   color='#f87171', linewidth=0.6, alpha=0.8)
    axes[0, 0].set_title('Noisy Waveform',   fontsize=10)
    axes[0, 0].set_xlabel('Time (s)')
    axes[0, 0].set_ylabel('Amplitude')

    axes[0, 1].plot(t_axis, y_clean, color='#34d399', linewidth=0.6, alpha=0.8)
    axes[0, 1].set_title('Denoised Waveform', fontsize=10)
    axes[0, 1].set_xlabel('Time (s)')

    D_noisy = librosa.amplitude_to_db(np.abs(librosa.stft(noisy)),   ref=np.max)
    D_clean = librosa.amplitude_to_db(np.abs(librosa.stft(y_clean)), ref=np.max)

    librosa.display.specshow(D_noisy, sr=SR, hop_length=512,
                             x_axis='time', y_axis='log', ax=axes[1, 0],
                             cmap='magma')
    axes[1, 0].set_title('Spectrogram – Noisy', fontsize=10)

    librosa.display.specshow(D_clean, sr=SR, hop_length=512,
                             x_axis='time', y_axis='log', ax=axes[1, 1],
                             cmap='magma')
    axes[1, 1].set_title('Spectrogram – Denoised', fontsize=10)

    plt.tight_layout()
    plt.show()

    print(f'✓ Denoising complete  |  signal length: {len(y_clean)/sr:.2f}s  |  sr: {sr} Hz')
    return y_clean, sr


# ── Run demo ──────────────────────────────────────────────────────────────────
y_demo, sr_demo = demo_denoising_on_synthetic()

---
## 2a · Stage 2 — Handcrafted Feature Extraction (Classical Stream)

Extracts interpretable acoustic biomarkers that clinical literature associates with Parkinson's dysarthria:

| Feature group | Clinical relevance |
|---|---|
| **MFCCs (13 × mean + std)** | Vocal tract shape / coarticulation breakdown |
| **F0 (mean, std, min, max)** | Fundamental frequency instability (jitter proxy) |
| **Spectral centroid (mean, std)** | Breathiness, high-frequency attenuation |
| **Spectral flux (mean, std)** | Shimmer / amplitude perturbation proxy |
| **Zero-crossing rate (mean, std)** | Voicing irregularity |
| **RMS energy (mean, std)** | Hypophonia (reduced loudness) |

In [ ]:
def extract_classical_features(
    y: np.ndarray,
    sr: int,
    n_mfcc: int = 13,
    f0_fmin: float = 50.0,
    f0_fmax: float = 500.0,
) -> Dict[str, float]:
    """
    Extract eGeMAPS-inspired handcrafted acoustic features.

    This constitutes the **Classical Stream** of VoxTriad's dual-stream
    feature extraction stage (Stage 2a).

    Features extracted
    ------------------
    - 13 MFCCs (mean + std)                  → 26 dims
    - Fundamental frequency F0 stats          →  4 dims
    - Spectral centroid (mean + std)          →  2 dims
    - Spectral flux     (mean + std)          →  2 dims
    - Zero-crossing rate (mean + std)         →  2 dims
    - RMS energy         (mean + std)         →  2 dims
                                        Total: 38 dims

    Args:
        y:       Mono waveform at `sr` Hz.
        sr:      Sample rate (Hz).
        n_mfcc:  Number of MFCC coefficients.
        f0_fmin: Minimum F0 search frequency (Hz).
        f0_fmax: Maximum F0 search frequency (Hz).

    Returns:
        Flat dict of {feature_name: float} suitable for DataFrame construction.
    """
    features: Dict[str, float] = {}

    # ── MFCCs ─────────────────────────────────────────────────────────────────
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc, n_fft=512, hop_length=160)
    for i in range(n_mfcc):
        features[f'mfcc_{i+1:02d}_mean'] = float(mfcc[i].mean())
        features[f'mfcc_{i+1:02d}_std']  = float(mfcc[i].std())

    # ── Fundamental frequency (F0) via pYIN ───────────────────────────────────
    f0, voiced_flag, _ = librosa.pyin(
        y, fmin=f0_fmin, fmax=f0_fmax, sr=sr,
        frame_length=1024, hop_length=256,
    )
    f0_voiced = f0[voiced_flag & ~np.isnan(f0)]
    if len(f0_voiced) > 1:
        features['f0_mean_hz']  = float(np.mean(f0_voiced))
        features['f0_std_hz']   = float(np.std(f0_voiced))
        features['f0_min_hz']   = float(np.min(f0_voiced))
        features['f0_max_hz']   = float(np.max(f0_voiced))
        # Jitter proxy: coefficient of variation of F0
        features['f0_jitter_cv'] = float(np.std(f0_voiced) / (np.mean(f0_voiced) + 1e-8))
    else:
        for k in ('f0_mean_hz', 'f0_std_hz', 'f0_min_hz', 'f0_max_hz', 'f0_jitter_cv'):
            features[k] = 0.0

    # ── Spectral centroid ─────────────────────────────────────────────────────
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=256)[0]
    features['spectral_centroid_mean'] = float(centroid.mean())
    features['spectral_centroid_std']  = float(centroid.std())

    # ── Spectral flux (shimmer proxy: frame-to-frame spectral change) ─────────
    onset_env = librosa.onset.onset_strength(y=y, sr=sr, hop_length=256)
    features['spectral_flux_mean'] = float(onset_env.mean())
    features['spectral_flux_std']  = float(onset_env.std())

    # ── Zero-crossing rate ────────────────────────────────────────────────────
    zcr = librosa.feature.zero_crossing_rate(y, hop_length=256)[0]
    features['zcr_mean'] = float(zcr.mean())
    features['zcr_std']  = float(zcr.std())

    # ── RMS energy (hypophonia marker) ────────────────────────────────────────
    rms = librosa.feature.rms(y=y, hop_length=256)[0]
    features['rms_mean'] = float(rms.mean())
    features['rms_std']  = float(rms.std())

    return features


# ── Run on demo signal ────────────────────────────────────────────────────────
classical_features = extract_classical_features(y_demo, sr_demo)

print(f'✓ Classical features extracted  |  dimensionality: {len(classical_features)}')
print()
for k, v in list(classical_features.items())[:12]:
    print(f'  {k:<35s}  {v:>10.5f}')
print('  ...')

---
## 2b · Stage 2 — Deep Embedding Extraction (wav2vec2 Stream)

Uses **facebook/wav2vec2-base** as a frozen feature encoder.
The model's transformer hidden states (768-dim per frame) are **mean-pooled** 
over time to produce a single fixed-size representation.

This 768-dim vector is the `embedding` column stored in the Supabase `features` table (pgvector).

In [ ]:
# ── Global model cache – loaded once per session ──────────────────────────────
_wav2vec2_processor: Optional[Wav2Vec2Processor] = None
_wav2vec2_model:     Optional[Wav2Vec2Model]     = None

WAV2VEC2_MODEL_ID = 'facebook/wav2vec2-base'
WAV2VEC2_EMBED_DIM = 768
WAV2VEC2_EXPECTED_SR = 16_000


def _load_wav2vec2() -> Tuple[Wav2Vec2Processor, Wav2Vec2Model]:
    """
    Lazily load and cache the wav2vec2 processor + model.
    Model is frozen (no gradient computation needed).
    """
    global _wav2vec2_processor, _wav2vec2_model

    if _wav2vec2_processor is None or _wav2vec2_model is None:
        print(f'  Loading {WAV2VEC2_MODEL_ID} from HuggingFace…', flush=True)
        t0 = time.time()

        _wav2vec2_processor = Wav2Vec2Processor.from_pretrained(WAV2VEC2_MODEL_ID)
        _wav2vec2_model     = Wav2Vec2Model.from_pretrained(WAV2VEC2_MODEL_ID)
        _wav2vec2_model.to(DEVICE)
        _wav2vec2_model.eval()   # inference-only, disable dropout

        for param in _wav2vec2_model.parameters():
            param.requires_grad_(False)  # freeze all weights

        print(f'  ✓ Model loaded in {time.time() - t0:.1f}s  |  device: {DEVICE}')

    return _wav2vec2_processor, _wav2vec2_model


def extract_deep_embeddings(
    y: np.ndarray,
    sr: int,
    max_duration_s: float = 10.0,
) -> np.ndarray:
    """
    Extract a mean-pooled 768-dim wav2vec2 embedding.

    Constitutes the **Deep Stream** of VoxTriad's dual-stream feature
    extraction stage (Stage 2b).

    The model expects 16 kHz mono audio.  Longer recordings are truncated
    to `max_duration_s` seconds to cap GPU memory usage.

    Args:
        y:               Mono waveform.
        sr:              Sample rate.  Must be 16 000 Hz.
        max_duration_s:  Maximum input duration to forward through the model.

    Returns:
        numpy array of shape (768,) – the session-level embedding vector.

    Raises:
        ValueError: If `sr` is not 16 000 Hz.
    """
    if sr != WAV2VEC2_EXPECTED_SR:
        raise ValueError(
            f'wav2vec2 requires 16 000 Hz audio, got {sr} Hz. '
            'Resample with librosa.resample() first.'
        )

    processor, model = _load_wav2vec2()

    # Truncate if necessary
    max_samples = int(max_duration_s * sr)
    y_trunc     = y[:max_samples] if len(y) > max_samples else y

    # Normalise to float32 [-1, 1]
    y_norm = y_trunc.astype(np.float32)
    peak   = np.abs(y_norm).max()
    if peak > 0:
        y_norm /= peak

    # Prepare input tensor
    inputs = processor(
        y_norm,
        sampling_rate=sr,
        return_tensors='pt',
        padding=True,
    )
    input_values = inputs.input_values.to(DEVICE)

    # Forward pass – extract last hidden states
    with torch.no_grad():
        outputs = model(input_values)

    # last_hidden_state: (batch=1, time_frames, 768)
    hidden_states = outputs.last_hidden_state          # (1, T, 768)
    embedding     = hidden_states.mean(dim=1).squeeze()  # (768,)

    return embedding.cpu().numpy()


# ── Run on demo signal ────────────────────────────────────────────────────────
print('Extracting deep embeddings from demo signal…')
embedding_demo = extract_deep_embeddings(y_demo, sr_demo)

print(f'✓ Embedding shape : {embedding_demo.shape}   (ready for pgvector storage)')
print(f'  L2 norm        : {np.linalg.norm(embedding_demo):.4f}')
print(f'  Value range    : [{embedding_demo.min():.4f}, {embedding_demo.max():.4f}]')

# Visualise the embedding as a heatmap
fig, ax = plt.subplots(figsize=(14, 1.8))
ax.imshow(embedding_demo[np.newaxis, :], aspect='auto', cmap='plasma')
ax.set_title('wav2vec2-base Mean-Pooled Embedding  (768 dims)', fontsize=10)
ax.set_xlabel('Embedding dimension')
ax.set_yticks([])
plt.tight_layout()
plt.show()

---
## 2c · Feature Fusion

Concatenate the classical and deep streams into a single unified feature vector.

In [ ]:
def fuse_features(
    classical: Dict[str, float],
    deep: np.ndarray,
) -> np.ndarray:
    """
    Late-fusion: concatenate classical and deep feature vectors.

    Args:
        classical: Dict from extract_classical_features().
        deep:      Array (768,) from extract_deep_embeddings().

    Returns:
        Fused feature vector of shape (len(classical) + 768,).
    """
    classical_vec = np.array(list(classical.values()), dtype=np.float32)
    deep_vec      = deep.astype(np.float32)
    return np.concatenate([classical_vec, deep_vec])


fused_demo = fuse_features(classical_features, embedding_demo)
print(f'✓ Fused feature vector  |  dims: {len(fused_demo)}')
print(f'  Classical dims  : {len(classical_features)}')
print(f'  Deep dims       : {len(embedding_demo)}')
print(f'  Total           : {len(fused_demo)}')

---
## 3 · Stage 3 — UCI Oxford Parkinson's Dataset Loading

Loads the clinical **UCI Oxford Parkinson's Dataset** (195 recordings, 31 subjects, 22 biomedical voice measures).

Design:
- **31 subjects** (8 healthy, 23 Parkinson's)
- **6 to 7 phonations per subject** (sustained vowels)
- **22 acoustic features**: MDVP metrics, jitter/shimmer perturbation quotients, NHR/HNR, and non-linear dynamical measures (RPDE, DFA, spread1/2, D2, PPE)
- Subject groupings extracted from recording IDs for **Leave-One-Subject-Out (LOSO) Cross-Validation** to strictly prevent speaker leakage

In [ ]:
# ── Load UCI Oxford Parkinson's Dataset ────────────────────────────────────────
import pandas as pd

csv_path = '../data/parkinsons.csv' if os.path.exists('../data/parkinsons.csv') else 'data/parkinsons.csv'
df = pd.read_csv(csv_path)

# Extract subject groupings from 'name' (e.g. 'phon_R01_S01_1' -> 'phon_R01_S01')
# Used for Leave-One-Subject-Out (LOSO) Cross-Validation to prevent data leakage
subject_groups = df['name'].apply(lambda x: '_'.join(x.split('_')[:3])).values

# Target label (status: 0 = Healthy, 1 = Parkinson's)
y_data = df['status'].values.astype(np.int32)

# Feature matrix X (drop 'name' and 'status')
feature_df = df.drop(columns=['name', 'status'])
feature_names = list(feature_df.columns)
X_data = feature_df.values.astype(np.float32)

print('=' * 55)
print("UCI Oxford Parkinson's Dataset Summary")
print('=' * 55)
print(f'  Total samples        : {len(X_data)}')
print(f'  Feature dimensions   : {X_data.shape[1]}')
print(f'  Healthy  (label=0)   : {(y_data == 0).sum()}')
print(f'  Parkinson (label=1)  : {(y_data == 1).sum()}')
print(f'  Unique subjects      : {len(np.unique(subject_groups))}')
print('=' * 55)

# ── Visualise class separation on 2 key features ──────────────────────────────
f0_col = feature_names.index('MDVP:Fo(Hz)')
jitter_col = feature_names.index('MDVP:Jitter(%)')

fig, ax = plt.subplots(figsize=(8, 5))
for label, colour, marker, name in [
    (0, '#34d399', 'o', 'Healthy'),
    (1, '#f87171', '^', "Parkinson's"),
]:
    mask = y_data == label
    ax.scatter(
        X_data[mask, f0_col], X_data[mask, jitter_col],
        c=colour, marker=marker, label=name, alpha=0.75, s=60, edgecolors='none',
    )
ax.set_xlabel('MDVP:Fo(Hz) (Fundamental Frequency)')
ax.set_ylabel('MDVP:Jitter(%) (Vocal Instability)')
ax.set_title("UCI Parkinson's Dataset · Class Separation", fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


---
## 3b · Stage 3 — Tri-Algorithmic Soft-Voting Meta-Ensemble

Three diverse base learners are combined via probability averaging (soft voting):

| Learner | Rationale |
|---|---|
| `SVC(rbf, probability=True)` | High-margin decision boundary; handles high-dim clinical features well |
| `RandomForestClassifier` | Bagging; robust to feature noise and missing values |
| `GradientBoostingClassifier` | Boosting; captures non-linear interactions between biomarkers |

In [ ]:
def build_meta_ensemble(
    n_estimators_rf: int = 200,
    n_estimators_gb: int = 150,
    svc_C: float = 1.0,
    gb_lr: float = 0.05,
    seed: int = SEED,
) -> Pipeline:
    """
    Build the Tri-Algorithmic Meta-Ensemble wrapped in a scikit-learn Pipeline.

    Pipeline stages:
      1. StandardScaler  – zero-mean, unit-variance normalisation
      2. VotingClassifier (soft) – SVC + RF + GBM

    Soft voting computes:  p_final = (p_SVC + p_RF + p_GBM) / 3
    This averaging reduces variance and is the foundation of the
    Evidential uncertainty estimation in the next cell.

    Args:
        n_estimators_rf: Trees in the Random Forest.
        n_estimators_gb: Boosting rounds for GBM.
        svc_C:           SVC regularisation parameter.
        gb_lr:           GBM learning rate.
        seed:            Random seed.

    Returns:
        Unfitted scikit-learn Pipeline.
    """
    # ── Base learners ─────────────────────────────────────────────────────────
    svc = SVC(
        kernel='rbf',
        C=svc_C,
        probability=True,      # needed for soft voting
        class_weight='balanced',
        random_state=seed,
    )

    rf = RandomForestClassifier(
        n_estimators=n_estimators_rf,
        max_features='sqrt',
        class_weight='balanced',
        n_jobs=-1,
        random_state=seed,
    )

    gb = GradientBoostingClassifier(
        n_estimators=n_estimators_gb,
        learning_rate=gb_lr,
        max_depth=4,
        subsample=0.8,
        random_state=seed,
    )

    # ── Soft-voting ensemble ──────────────────────────────────────────────────
    ensemble = VotingClassifier(
        estimators=[
            ('svc', svc),
            ('rf',  rf),
            ('gb',  gb),
        ],
        voting='soft',         # average class probabilities
        n_jobs=1,              # VotingClassifier spawns sub-jobs internally
    )

    # ── Full pipeline with z-score normalisation ──────────────────────────────
    pipeline = Pipeline([
        ('scaler',   StandardScaler()),
        ('ensemble', ensemble),
    ])

    return pipeline


# ── Instantiate ───────────────────────────────────────────────────────────────
model = build_meta_ensemble()

print('✓ Meta-ensemble pipeline built')
print()
print(model)

---
## 3c · Epistemic Uncertainty — Evidential Scoring

Computes **Shannon entropy** of the ensemble's predicted probability distribution 
as a proxy for the Dirichlet-based evidential uncertainty score.

$$H(p) = -\sum_{c} p_c \log_2 p_c$$

- `H = 0` → maximum confidence (all probability mass on one class)
- `H = 1` → maximum uncertainty (uniform distribution across classes)

High-uncertainty samples are flagged for clinician review instead of being auto-classified.

In [ ]:
def calculate_epistemic_uncertainty(
    probs: np.ndarray,
    epsilon: float = 1e-12,
) -> np.ndarray:
    """
    Compute normalised Shannon entropy as an epistemic uncertainty score.

    Acts as a Dirichlet-based evidential uncertainty measure:
    high entropy → the ensemble's internal distributions conflict,
    signalling a sample that lies near the decision boundary.

    Formula:
        H(p) = -Σ p_c * log2(p_c)   for c in {0, 1, …, C-1}

    Normalised by log2(C) so the output is in [0, 1].

    Args:
        probs:   Probability matrix of shape (n_samples, n_classes),
                 as returned by `ensemble.predict_proba(X)`.
        epsilon: Small value added to probabilities before log to prevent
                 log(0) = -∞.

    Returns:
        Array of shape (n_samples,) with uncertainty scores in [0, 1].
        Values ≥ 0.7 are conventionally flagged for clinical review.
    """
    probs = np.clip(probs, epsilon, 1.0)
    n_classes = probs.shape[1]
    entropy = -np.sum(probs * np.log2(probs), axis=1)
    # Normalise: max possible entropy is log2(n_classes)
    return entropy / np.log2(n_classes) if n_classes > 1 else entropy


def interpret_uncertainty(score: float) -> str:
    """Map a scalar uncertainty score to a clinical review tier."""
    if score < 0.35:
        return '🟢 High confidence'
    elif score < 0.65:
        return '🟡 Moderate uncertainty – review recommended'
    else:
        return '🔴 High uncertainty – mandatory clinician review'


# ── Quick demonstration on toy probabilities ──────────────────────────────────
toy_probs = np.array([
    [0.97, 0.03],    # very confident Healthy
    [0.55, 0.45],    # borderline
    [0.50, 0.50],    # maximum uncertainty
    [0.02, 0.98],    # very confident PD
], dtype=np.float32)

toy_uncertainty = calculate_epistemic_uncertainty(toy_probs)

print('Epistemic Uncertainty Demonstration')
print('─' * 60)
print(f'  {"p(Healthy)":>12}  {"p(PD)":>8}  {"H (norm)":>10}  Tier')
print('─' * 60)
for p, u in zip(toy_probs, toy_uncertainty):
    tier = interpret_uncertainty(u)
    print(f'  {p[0]:>12.2f}  {p[1]:>8.2f}  {u:>10.4f}  {tier}')
print('─' * 60)

# Visualise
fig, ax = plt.subplots(figsize=(8, 3))
colours = ['#34d399' if u < 0.35 else ('#facc15' if u < 0.65 else '#f87171')
           for u in toy_uncertainty]
bars = ax.bar(range(len(toy_uncertainty)), toy_uncertainty, color=colours, width=0.5, edgecolor='none')
ax.axhline(0.35, color='#facc15', linestyle='--', alpha=0.7, label='Review threshold')
ax.axhline(0.65, color='#f87171', linestyle='--', alpha=0.7, label='Mandatory review')
ax.set_xticks(range(len(toy_uncertainty)))
ax.set_xticklabels([f'Sample {i+1}' for i in range(len(toy_uncertainty))])
ax.set_ylabel('Normalised Entropy H')
ax.set_ylim(0, 1.05)
ax.set_title('Epistemic Uncertainty Scores', fontweight='bold')
ax.legend(loc='upper left', fontsize=8)
ax.grid(True, axis='y', alpha=0.4)
plt.tight_layout()
plt.show()

---
## 4 · Validation — Leave-One-Subject-Out Cross-Validation (LOSO-CV)

**LOSO-CV** is the gold standard validation strategy for clinical speech datasets:
- Each fold holds out **all recordings from one subject**
- Prevents **data leakage** between sessions from the same speaker
- Measures true **generalisation across new patients** (realistic clinical scenario)

$$\text{LOSO-Acc} = \frac{1}{N_s} \sum_{s=1}^{N_s} \text{Acc}_{\text{fold } s}$$

In [ ]:
def run_loso_cv(
    X: np.ndarray,
    y: np.ndarray,
    groups: np.ndarray,
    pipeline: Pipeline,
    uncertainty_threshold: float = 0.65,
) -> dict:
    """
    Run Leave-One-Subject-Out Cross-Validation.

    For each fold:
      1. Train on all subjects except the left-out subject
      2. Predict on the held-out subject's recordings
      3. Compute epistemic uncertainty on the held-out predictions

    Args:
        X:                     Feature matrix.
        y:                     Ground-truth labels.
        groups:                Subject IDs for grouping.
        pipeline:              Unfitted scikit-learn Pipeline.
        uncertainty_threshold: Samples above this entropy are flagged.

    Returns:
        Results dict containing per-fold metrics and aggregated scores.
    """
    logo         = LeaveOneGroupOut()
    n_splits     = logo.get_n_splits(X, y, groups)
    unique_subjs = np.unique(groups)

    fold_accs   = []
    fold_f1s    = []
    fold_aucs   = []
    fold_unc    = []
    all_y_true  = []
    all_y_pred  = []
    all_y_prob  = []

    print(f'Running LOSO-CV  |  {n_splits} folds  |  {len(X)} samples')
    print('─' * 70)
    print(f'  {"Fold":>5}  {"Subject":>7}  {"n_test":>6}  {"Acc":>7}  {"F1":>7}  {"AUC":>7}  {"Unc (mean)":>11}')
    print('─' * 70)

    from copy import deepcopy

    for fold_idx, (train_idx, test_idx) in enumerate(
        logo.split(X, y, groups)
    ):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        subj_id         = groups[test_idx[0]]

        # Fresh copy of the pipeline for each fold
        fold_model = deepcopy(pipeline)
        fold_model.fit(X_train, y_train)

        # Predictions
        y_pred = fold_model.predict(X_test)
        y_prob = fold_model.predict_proba(X_test)   # (n_test, 2)

        # Per-fold metrics
        acc = accuracy_score(y_test, y_pred)
        f1  = f1_score(y_test, y_pred, average='binary', zero_division=0)
        # AUC requires at least two classes in the test fold
        try:
            auc = roc_auc_score(y_test, y_prob[:, 1])
        except ValueError:
            auc = float('nan')

        # Epistemic uncertainty for the fold
        unc = calculate_epistemic_uncertainty(y_prob)
        mean_unc = unc.mean()
        flagged  = (unc >= uncertainty_threshold).sum()

        fold_accs.append(acc)
        fold_f1s.append(f1)
        fold_aucs.append(auc)
        fold_unc.append(mean_unc)

        all_y_true.extend(y_test.tolist())
        all_y_pred.extend(y_pred.tolist())
        all_y_prob.extend(y_prob.tolist())

        print(
            f'  {fold_idx+1:>5}  '
            f'Subj-{subj_id:02d}  '
            f'{len(y_test):>6}  '
            f'{acc:>7.3f}  '
            f'{f1:>7.3f}  '
            f'{auc:>7.3f}  '
            f'{mean_unc:>11.4f}  '
            f'({flagged} flagged)'
        )

    all_y_true = np.array(all_y_true)
    all_y_pred = np.array(all_y_pred)
    all_y_prob = np.array(all_y_prob)

    overall_acc = accuracy_score(all_y_true, all_y_pred)
    overall_f1  = f1_score(all_y_true, all_y_pred, average='binary', zero_division=0)
    try:
        overall_auc = roc_auc_score(all_y_true, all_y_prob[:, 1])
    except ValueError:
        overall_auc = float('nan')

    print('─' * 70)
    print(f'  Mean fold Acc : {np.nanmean(fold_accs):.3f}  ±{np.nanstd(fold_accs):.3f}')
    print(f'  Mean fold F1  : {np.nanmean(fold_f1s):.3f}  ±{np.nanstd(fold_f1s):.3f}')
    print(f'  Mean fold AUC : {np.nanmean(fold_aucs):.3f}  ±{np.nanstd(fold_aucs):.3f}')
    print(f'  Overall Acc   : {overall_acc:.3f}')
    print(f'  Overall F1    : {overall_f1:.3f}')
    print(f'  Overall AUC   : {overall_auc:.3f}')
    print('─' * 70)

    return {
        'fold_accs': fold_accs,
        'fold_f1s':  fold_f1s,
        'fold_aucs': fold_aucs,
        'fold_unc':  fold_unc,
        'overall_acc': overall_acc,
        'overall_f1':  overall_f1,
        'overall_auc': overall_auc,
        'y_true': all_y_true,
        'y_pred': all_y_pred,
        'y_prob': all_y_prob,
    }


# ── Execute LOSO-CV ───────────────────────────────────────────────────────────
cv_results = run_loso_cv(X_data, y_data, subject_groups, model)

---
## 5 · Results Visualisation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('VoxTriad · LOSO-CV Results Summary', fontsize=13, fontweight='bold')

# ── 1. Per-fold performance bars ──────────────────────────────────────────────
n_folds = len(cv_results['fold_accs'])
x_folds = np.arange(n_folds)
w = 0.28

axes[0].bar(x_folds - w,   cv_results['fold_accs'], width=w, label='Accuracy',  color='#6366f1', alpha=0.85)
axes[0].bar(x_folds,       cv_results['fold_f1s'],  width=w, label='F1',        color='#34d399', alpha=0.85)
axes[0].bar(x_folds + w,   cv_results['fold_aucs'], width=w, label='AUC',       color='#f59e0b', alpha=0.85)
axes[0].axhline(cv_results['overall_acc'], color='#6366f1', linestyle='--', alpha=0.6)
axes[0].set_xlabel('Fold (Left-out subject)')
axes[0].set_ylabel('Score')
axes[0].set_title('Per-Fold Metrics')
axes[0].set_ylim(0, 1.15)
axes[0].legend(fontsize=8)
axes[0].set_xticks(x_folds)
axes[0].set_xticklabels([f'S{i}' for i in range(n_folds)], fontsize=7)
axes[0].grid(True, axis='y', alpha=0.3)

# ── 2. Confusion matrix ───────────────────────────────────────────────────────
cm = confusion_matrix(cv_results['y_true'], cv_results['y_pred'])
cm_display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Healthy', "Parkinson's"])
cm_display.plot(ax=axes[1], colorbar=False, cmap='Blues')
axes[1].set_title('Confusion Matrix (LOSO-CV)')
axes[1].set_facecolor('#1e293b')

# ── 3. Uncertainty distribution ───────────────────────────────────────────────
all_uncertainty = calculate_epistemic_uncertainty(cv_results['y_prob'])
axes[2].hist(all_uncertainty[cv_results['y_true'] == 0], bins=15,
             color='#34d399', alpha=0.7, label='Healthy', edgecolor='none')
axes[2].hist(all_uncertainty[cv_results['y_true'] == 1], bins=15,
             color='#f87171', alpha=0.7, label="Parkinson's", edgecolor='none')
axes[2].axvline(0.65, color='#facc15', linestyle='--', label='Review threshold')
axes[2].set_xlabel('Epistemic Uncertainty H')
axes[2].set_ylabel('Count')
axes[2].set_title('Uncertainty Distribution')
axes[2].legend(fontsize=8)
axes[2].grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# ── Classification report ─────────────────────────────────────────────────────
print('\nDetailed Classification Report')
print('─' * 50)
print(classification_report(
    cv_results['y_true'],
    cv_results['y_pred'],
    target_names=['Healthy', "Parkinson's"],
))

# ── Flagged samples ───────────────────────────────────────────────────────────
n_flagged = (all_uncertainty >= 0.65).sum()
print(f'Samples flagged for mandatory review (H ≥ 0.65): {n_flagged}/{len(all_uncertainty)} '
      f'({100 * n_flagged / len(all_uncertainty):.1f}%)')

---
## 6 · Pipeline Summary & Phase 3 Roadmap

### What was built in Phase 2

| Component | Status | Location |
|---|---|---|
| PPD-Net denoising (spectral gate) | ✅ | `apply_denoising()` |
| Classical feature extraction | ✅ | `extract_classical_features()` |
| wav2vec2 deep embeddings | ✅ | `extract_deep_embeddings()` |
| Feature fusion | ✅ | `fuse_features()` |
| Soft-voting meta-ensemble | ✅ | `build_meta_ensemble()` |
| Epistemic uncertainty scoring | ✅ | `calculate_epistemic_uncertainty()` |
| LOSO-CV validation | ✅ | `run_loso_cv()` |

### Phase 3 TODO

- [ ] **Supabase integration**: migrate `extract_*` functions to `backend/feature_extraction.py` and store results in the `features` table with `pgvector` embedding
- [ ] **Real dataset**: replace synthetic data with [mPower](https://www.synapse.org/#!Synapse:syn4993293) or [PC-GITA](https://www.kaggle.com/datasets/jcardenas14/pc-gita-corpus)
- [ ] **Hyperparameter search**: `Optuna` Bayesian optimisation over SVC-C, RF depth, GBM lr
- [ ] **Learned denoising**: fine-tune a denoising U-Net (e.g. CleanUNet) to replace spectral gating
- [ ] **Gemini LLM report**: pipe high-uncertainty cases to Gemini 1.5 Pro for narrative clinical summary
- [ ] **SHAP explainability**: produce per-prediction feature attribution heatmaps

In [ ]:
print('=' * 60)
print('VoxTriad Phase 2 · Pipeline Complete')
print('=' * 60)
print(f'  Overall Accuracy  : {cv_results["overall_acc"]:.3f}')
print(f'  Overall F1-Score  : {cv_results["overall_f1"]:.3f}')
print(f'  Overall AUC-ROC   : {cv_results["overall_auc"]:.3f}')
print(f'  Compute device    : {DEVICE}')
print('=' * 60)